# 🧠 تدريب الذكاء الاصطناعي لبروتوكول BARH
## نموذج LSTM + Deep Q-Learning للشبكات الذكية

---

### 📋 خطة التدريب:
1. **إعداد البيئة** - تثبيت المكتبات والأدوات
2. **توليد البيانات** - إنشاء بيانات تدريب واقعية
3. **تدريب LSTM** - للتنبؤ بالتصحيحات المطلوبة
4. **تدريب DQN** - لاختيار الإجراءات المثلى
5. **التقييم والاختبار** - قياس الأداء والدقة
6. **الدمج والتطبيق** - إنشاء النظام المتكامل

## 🔧 المرحلة 1: إعداد البيئة

In [ ]:
# تثبيت المكتبات الأساسية
!pip install tensorflow==2.13.0
!pip install torch torchvision torchaudio
!pip install scikit-learn pandas numpy matplotlib seaborn
!pip install wandb tensorboard
!pip install networkx

In [ ]:
# استيراد المكتبات
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import time
from datetime import datetime, timedelta
from collections import deque
import random
import warnings
warnings.filterwarnings('ignore')

# TensorFlow/Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# Scikit-learn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# فحص GPU
print("TensorFlow GPU Available:", tf.config.list_physical_devices('GPU'))
print("PyTorch GPU Available:", torch.cuda.is_available())

# إعداد البذرة للتكرار
np.random.seed(42)
tf.random.set_seed(42)
torch.manual_seed(42)

print("✅ تم إعداد البيئة بنجاح!")

In [ ]:
# إنشاء هيكل المشروع
import os

directories = [
    '/content/BARH-AI/data/raw',
    '/content/BARH-AI/data/processed', 
    '/content/BARH-AI/data/synthetic',
    '/content/BARH-AI/models/lstm',
    '/content/BARH-AI/models/dqn',
    '/content/BARH-AI/models/checkpoints',
    '/content/BARH-AI/results',
    '/content/BARH-AI/logs'
]

for directory in directories:
    os.makedirs(directory, exist_ok=True)
    
print("✅ تم إنشاء هيكل المشروع!")

## 📊 المرحلة 2: توليد البيانات

In [ ]:
class NetworkDataGenerator:
    """مولد البيانات الاصطناعية للشبكات"""
    
    def __init__(self, seed=42):
        np.random.seed(seed)
        
    def generate_network_scenarios(self, num_samples=50000):
        """توليد سيناريوهات شبكة متنوعة"""
        
        print(f"🔄 توليد {num_samples:,} عينة من بيانات الشبكة...")
        
        scenarios = []
        for i in range(num_samples):
            if i % 10000 == 0:
                print(f"   تم توليد {i:,} عينة...")
                
            # معاملات الشبكة الأساسية
            base_latency = np.random.normal(50, 15)
            base_throughput = np.random.normal(100, 25)
            base_packet_loss = np.random.exponential(0.02)
            
            # إضافة تشويش وأنماط مختلفة
            time_of_day = (i % 1440) / 1440  # دورة 24 ساعة
            congestion_factor = 1 + 0.5 * np.sin(2 * np.pi * time_of_day)
            
            # حالات الشبكة المختلفة
            network_state = np.random.choice(['normal', 'congested', 'unstable', 'peak'], 
                                           p=[0.4, 0.3, 0.2, 0.1])
            
            if network_state == 'congested':
                latency = base_latency * congestion_factor * np.random.uniform(1.5, 3.0)
                throughput = base_throughput / congestion_factor * np.random.uniform(0.3, 0.7)
                packet_loss = base_packet_loss * np.random.uniform(2.0, 5.0)
            elif network_state == 'unstable':
                latency = base_latency * np.random.uniform(0.8, 2.5)
                throughput = base_throughput * np.random.uniform(0.4, 1.2)
                packet_loss = base_packet_loss * np.random.uniform(1.0, 8.0)
            elif network_state == 'peak':
                latency = base_latency * np.random.uniform(2.0, 4.0)
                throughput = base_throughput * np.random.uniform(0.2, 0.5)
                packet_loss = base_packet_loss * np.random.uniform(3.0, 10.0)
            else:  # normal
                latency = base_latency * np.random.uniform(0.8, 1.2)
                throughput = base_throughput * np.random.uniform(0.9, 1.1)
                packet_loss = base_packet_loss * np.random.uniform(0.5, 1.5)
            
            # إضافة معاملات إضافية
            cpu_usage = np.random.beta(2, 5) * 100
            memory_usage = np.random.beta(3, 4) * 100
            bandwidth_utilization = np.random.beta(2, 3) * 100
            
            scenarios.append({
                'timestamp': i,
                'latency': max(1, latency),
                'throughput': max(1, throughput),
                'packet_loss': max(0, min(100, packet_loss)),
                'cpu_usage': cpu_usage,
                'memory_usage': memory_usage,
                'bandwidth_utilization': bandwidth_utilization,
                'network_state': network_state,
                'time_of_day': time_of_day,
                'congestion_factor': congestion_factor
            })
            
        print("✅ تم توليد البيانات بنجاح!")
        return pd.DataFrame(scenarios)
    
    def add_correction_labels(self, df):
        """إضافة تسميات التصحيح المطلوبة"""
        
        print("🏷️ إضافة تسميات التصحيح...")
        
        corrections = []
        
        for _, row in df.iterrows():
            correction = {
                'buffer_optimization': 0,
                'connection_pooling': 0,
                'route_optimization': 0,
                'adaptive_compression': 0,
                'parallel_connections': 0,
                'predictive_prefetch': 0
            }
            
            # قواعد التصحيح بناءً على حالة الشبكة
            if row['latency'] > 80:
                correction['route_optimization'] = 1
                correction['connection_pooling'] = 1
                
            if row['packet_loss'] > 5:
                correction['adaptive_compression'] = 1
                correction['buffer_optimization'] = 1
                
            if row['throughput'] < 50:
                correction['parallel_connections'] = 1
                correction['predictive_prefetch'] = 1
                
            if row['network_state'] == 'unstable':
                correction['buffer_optimization'] = 1
                correction['adaptive_compression'] = 1
                
            corrections.append(correction)
            
        correction_df = pd.DataFrame(corrections)
        result = pd.concat([df, correction_df], axis=1)
        
        print("✅ تم إضافة التسميات بنجاح!")
        return result

# توليد البيانات
generator = NetworkDataGenerator()
data = generator.generate_network_scenarios(50000)
labeled_data = generator.add_correction_labels(data)

# حفظ البيانات
labeled_data.to_csv('/content/BARH-AI/data/synthetic/network_data.csv', index=False)

print(f"📊 تم إنشاء {len(labeled_data):,} عينة تدريب")
print(f"📁 تم حفظ البيانات في: /content/BARH-AI/data/synthetic/network_data.csv")

In [ ]:
# فحص جودة البيانات
def validate_data_quality(df):
    """فحص جودة البيانات"""
    
    print("=== 📋 تقرير جودة البيانات ===")
    print(f"عدد العينات: {len(df):,}")
    print(f"عدد الأعمدة: {len(df.columns)}")
    print(f"القيم المفقودة: {df.isnull().sum().sum()}")
    print(f"القيم المكررة: {df.duplicated().sum()}")
    
    # فحص التوزيعات
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    print("\n📊 فحص القيم الشاذة:")
    for col in numeric_cols[:6]:  # أهم 6 أعمدة
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        outliers = ((df[col] < (q1 - 1.5 * iqr)) | (df[col] > (q3 + 1.5 * iqr))).sum()
        print(f"  {col}: {outliers:,} قيمة شاذة ({outliers/len(df)*100:.1f}%)")
    
    # إحصائيات التصحيحات
    correction_cols = ['buffer_optimization', 'connection_pooling', 'route_optimization',
                      'adaptive_compression', 'parallel_connections', 'predictive_prefetch']
    
    print("\n🔧 إحصائيات التصحيحات:")
    for col in correction_cols:
        count = df[col].sum()
        percentage = count / len(df) * 100
        print(f"  {col}: {count:,} ({percentage:.1f}%)")
    
    return True

validate_data_quality(labeled_data)

In [ ]:
# رسم توزيع البيانات
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('توزيع معاملات الشبكة', fontsize=16, fontweight='bold')

# المعاملات الأساسية
params = ['latency', 'throughput', 'packet_loss', 'cpu_usage', 'memory_usage', 'bandwidth_utilization']
colors = ['red', 'green', 'blue', 'orange', 'purple', 'brown']

for i, (param, color) in enumerate(zip(params, colors)):
    row, col = i // 3, i % 3
    axes[row, col].hist(labeled_data[param], bins=50, alpha=0.7, color=color, edgecolor='black')
    axes[row, col].set_title(f'{param.replace("_", " ").title()}')
    axes[row, col].set_xlabel('القيمة')
    axes[row, col].set_ylabel('التكرار')
    axes[row, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/BARH-AI/results/data_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("📈 تم حفظ رسم التوزيع في: /content/BARH-AI/results/data_distribution.png")